# Mistake detection: CocoChorales-E2 and CocoChorales-E

Isolated and end-to-end benchmarks, with separate results and provenance:

- **CocoChorales-E2**: our monophonic injection pipeline, rendered with FluidSynth / MuseScore.
- **CocoChorales-E**: the authors’ performed MIDI and unchanged mistake labels, with both score and performance audio re-rendered using FluidSynth / `resources/MuseScore_General.sf3`. Overlapping notes remain overlapping.

**Run All loads the completed isolated results and evaluates the full end-to-end cohorts: 260 tracks per dataset**. Saved isolated results retain their original code/settings; they are not rerun with the current implementation. E2 also evaluates 260 clean controls. Completed compatible checkpoints are reused; interrupted runs resume missing evaluations. The audio cells retain fixed output folders and do not force pitch detection. Historical folder names containing `distinct_pieces` are retained for asset reuse; selection now includes all stems, including those sharing a source piece. Audio benchmarks run in fresh subprocesses. CPU preparation is parallel; neural models use persistent, RAM-bounded worker pools. Use the Attune `at-venv` kernel and the existing [model setup instructions](../modules/mistake/README.md).

The primary measure is pooled missed/extra-event F1 at 100 ms / 50 cents; substitutions count as missed + extra. This is not note accuracy. Saved runs retain their historical code/settings. The two datasets are not paired and their scores should not be treated as a controlled comparison.

Rendering evidence and listening examples live in [coco-e_audit.ipynb](coco-e_audit.ipynb). Historical ablations are preserved in [archive](archive/mistake_experiments_2026-10-04.ipynb).

**Historical October 8 promotion (superseded by the November 17 defaults below):** offline pitch tracking uses a fixed 0.990 HMM gate and a separate 0.9875 final voicing gate, with the 0.04 × p95(RMS) floor. Shared pitch cache version 15 invalidates older stages. Restart the kernel before running this notebook. Audio benchmarks write to new promotion-specific folders; existing results are preserved. Both E and E2 use the current production pitch flow.


**Production sweep defaults (2026-11-17):** Run All refreshes only Attune, including the symbolic baseline, and combines its results with the original cached competitors. Exact evaluation cohorts and rendered native-E audio are retained. New reports are separate from historical runs; missing competitor coverage stops execution.


In [1]:
# autoreload when modules edited
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    (
        p
        for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (p / "app.py").is_file()
    )
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
RESULTS = REPO_ROOT / "benchmarks/results"
WORKERS = max(1, (os.cpu_count() or 4) - 1)
NEURAL_WORKERS = None  # Full RAM-bounded pool per algorithm; PolyTune then LadderSym.
RENDER_WORKERS = WORKERS
INJECTION_SEEDS = (0,)
COHORT_TAG = "260_tracks_annotations_v2_2026-10-07"
COHORT_MANIFESTS = {
    d: RESULTS / f"coco_{d.lower()}_isolated_{COHORT_TAG}" / "manifest.json"
    for d in ("E", "E2")
}
E_BASE = RESULTS / "native_v5_min100_seed0_pieces3/coco"
E2_OUTPUT = E_OUTPUT = None  # Set by each audio benchmark cell.
METHODS = ("Attune (repeat only)", "PolyTune", "LadderSym")


def pooled_table(frame):
    if frame.duplicated(["method", "case_id"]).any():
        raise ValueError("Duplicate case/method rows")
    table = frame.groupby("method").agg(
        Cases=("case_id", "nunique"),
        TP=("tp", "sum"),
        FP=("fp", "sum"),
        FN=("fn", "sum"),
    )
    denominator = 2 * table.TP + table.FP + table.FN
    table["F1 %"] = 200 * table.TP / denominator.where(denominator.ne(0))
    return table.round(2)

from benchmarks.notebooks.scripts.mistake_default_refresh import refresh


## Isolated mistake detection — 260 source tracks per dataset

These MIDI-only runs compare Attune's alignment component with Nakamura and Parangonar Automatic, DualDTW, and TheGlueNote. Both cohorts contain **20 test tracks per instrument (260 total)**. Selection is frozen before evaluation and fails if any instrument quota is unavailable.

- **E:** the pinned author-mirror score and error labels. Exact performed notes are reconstructed from correct + extra label MIDI, retaining overlaps and simultaneous notes. Raw performed MIDI can contain unlabelled events; discrepancies are saved in each case’s `annotation_audit.json`. No rendering is needed for exact-note evaluation.
- **E2:** the existing monophonic injection procedure, with one injection seed (0) at 25% injection (**260 altered cases**) and one clean control per source (**260 controls**).

Attune uses production edit costs, an endpoint timing fit and one robust matched-onset refit through an event-list adapter; no note extraction or repeat recovery is applied. All aligners receive every MIDI event. Native E retains author-label timestamp semantics; E2 uses the generated net-error labels. The cohorts are separate, not paired with each other.

The isolated execution cell is disabled by default (`RUN_ISOLATED = False`): Run All reads the completed historical results. Enable it only for an intentional isolated run, using a fresh output tag if code/configuration has changed. E preparation can download MIDI/labels from the pinned mirror; E2 requires the local CocoChorales test shards. No audio or audio-model inference is performed. Use a new tag after changing code/configuration; interrupted matching runs resume per-case checkpoints.

Progress uses one updating line, e.g. `E1 [ 28/2600]: Attune: 231396/0_violin`. The global total is tracks × datasets × selected methods: 260 × 2 × 5 = 2,600 by default. Each track–method combination completes after all its seeds/conditions, including E2 controls. The counter continues across datasets and never resets when the displayed method changes. Workers default to available cores with a memory cap for matchers; set `ISOLATED_WORKERS` to override. Worker chatter goes to per-case logs. The single-seed output tag preserves earlier runs; the original v2 manifests retain the previously tested source-track lists.


In [3]:
from benchmarks.modules.mistake.MistakeBenchmarker import MistakeBenchmarker

RUN_ISOLATED = False  # Read saved symbolic results; do not launch inference on Run All.
ISOLATED_TAG = "260_tracks_annotations_seed0_2026-10-07"
ISOLATED_WORKERS = None  # Auto: available CPU cores, capped by matcher memory.
ISOLATED_OUTPUTS = {d: RESULTS / f"coco_{d.lower()}_isolated_{ISOLATED_TAG}" for d in ("E", "E2")}
ISOLATED_ROOTS = {
    "E": REPO_ROOT / "benchmarks/datasets/native_errors/coco",
    "E2": REPO_ROOT / "benchmarks/datasets/cocochorales_tiny",
}


In [4]:
# Refresh symbolic Attune only; preserve all saved competitor rows.
ISOLATED_OUTPUTS = {d: refresh(output, mode="symbolic", workers=WORKERS)
                    for d, output in ISOLATED_OUTPUTS.items()}


In [5]:
for dataset, output in ISOLATED_OUTPUTS.items():
    if (output / "rows.csv").exists():
        tables = MistakeBenchmarker.isolated_tables(output)
        display(tables["errors"])
        if not tables["clean"].empty:
            display(tables["clean"])
    else:
        print(f"{dataset}: no isolated results yet; run the isolated benchmark cell above.")

if all((p / "rows.csv").exists() for p in ISOLATED_OUTPUTS.values()):
    display(MistakeBenchmarker.isolated_average_table(ISOLATED_OUTPUTS).round(2))


,dataset,method,tracks,cases,tp,fp,fn,precision,recall,f1
0,E,Attune (no refinement),260,260,505,14,11,97.303,97.868,97.585
1,E,Nakamura,260,260,495,30,21,94.286,95.930,95.101
3,E,Parangonar DualDTW,260,260,441,70,75,86.301,85.465,85.881
2,E,Parangonar Automatic,260,260,429,92,87,82.342,83.140,82.739
4,E,Parangonar TheGlueNote,260,260,425,276,91,60.628,82.364,69.844


,dataset,method,tracks,cases,tp,fp,fn,precision,recall,f1
5,E2,Attune (no refinement),260,260,649,6,6,99.084,99.084,99.084
8,E2,Parangonar DualDTW,260,260,634,5,21,99.218,96.794,97.991
7,E2,Parangonar Automatic,260,260,585,84,70,87.444,89.313,88.369
6,E2,Nakamura,260,260,646,175,9,78.685,98.626,87.534
9,E2,Parangonar TheGlueNote,260,260,620,297,35,67.612,94.656,78.880


,dataset,method,tracks,cases,fp,clean_fp_per_100_notes
0,E2,Attune (no refinement),260,260,0,0.000
1,E2,Nakamura,260,260,0,0.000
2,E2,Parangonar Automatic,260,260,8,0.130
3,E2,Parangonar DualDTW,260,260,0,0.000
4,E2,Parangonar TheGlueNote,260,260,76,1.232


,precision,recall,f1
method,,,
Attune (no refinement),98.19,98.48,98.33
Parangonar DualDTW,92.76,91.13,91.94
Nakamura,86.49,97.28,91.32
Parangonar Automatic,84.89,86.23,85.55
Parangonar TheGlueNote,64.12,88.51,74.36


### Statistical significance — paired permutation test with Holm correction

The primary statistic is the difference in **pooled missed/extra-event F1 at 100 ms / 50 cents**. We use a two-sided paired permutation test with **9,999 swaps** (exact enumeration for sufficiently few source groups), swapping complete TP/FP/FN vectors and recomputing pooled F1. All seeds and instrument stems from the same source piece are swapped together, avoiding treatment of dependent performances as independent samples.

Holm correction covers one family of **eight comparisons**: Attune versus four aligners, separately on E and E2, at α = 0.05. Positive `difference_pp` favors Attune. Every selected case and method must be complete; missing pairings stop the calculation. E2 clean controls are reported separately. The 50/200 ms scores are descriptive secondary results, outside this prespecified test family. These are synthetic test cohorts; any overlap with prior tuning should be disclosed when reporting generalization.

This cell only reads completed isolated results and computes statistics; it never launches a benchmark.


In [6]:
if all((p / "rows.csv").exists() for p in ISOLATED_OUTPUTS.values()):
    significance, coverage = MistakeBenchmarker.isolated_significance(
        ISOLATED_OUTPUTS, tolerance=0.1, n_resamples=9999, seed=0, alpha=0.05,
    )
    display(coverage)
    display(significance)
else:
    print("Complete both isolated datasets before calculating significance.")


,dataset,tracks,cases,source_groups,methods,excluded_cases
0,E,260,260,98,5,0
1,E2,260,260,256,5,0


,dataset,method,tolerance,tracks,cases,source_groups,attune_f1,competitor_f1,difference_pp,p_value,permutations,exact,seed,p_holm,significant,alpha,holm_family_size
0,E,Parangonar DualDTW,0.1,260,260,98,97.584541,85.881207,11.703334,0.0001,9999,False,0,0.0008,True,0.05,8
1,E,Parangonar Automatic,0.1,260,260,98,97.584541,82.738669,14.845872,0.0001,9999,False,0,0.0008,True,0.05,8
2,E,Parangonar TheGlueNote,0.1,260,260,98,97.584541,69.843878,27.740663,0.0001,9999,False,0,0.0008,True,0.05,8
3,E,Nakamura,0.1,260,260,98,97.584541,95.100865,2.483677,0.1410,9999,False,0,0.1410,False,0.05,8
4,E2,Parangonar DualDTW,0.1,260,260,256,99.083969,97.990726,1.093243,0.0261,9999,False,0,0.0522,False,0.05,8
5,E2,Parangonar Automatic,0.1,260,260,256,99.083969,88.368580,10.715389,0.0001,9999,False,0,0.0008,True,0.05,8
6,E2,Parangonar TheGlueNote,0.1,260,260,256,99.083969,78.880407,20.203562,0.0001,9999,False,0,0.0008,True,0.05,8
7,E2,Nakamura,0.1,260,260,256,99.083969,87.533875,11.550094,0.0001,9999,False,0,0.0008,True,0.05,8


In [7]:
# Combined symbolic results: CocoChorales-E (E1) + E2, 100 ms / 50 cents.
# Equal-weight mean of dataset-pooled metrics, matching isolated_average_table.
# Clean controls are excluded. Each dataset/source-piece cluster is swapped as
# a unit, retaining all its stems and seeds. Four combined comparisons form
# their own Holm family, separate from the eight per-dataset tests above.
import itertools
import numpy as np
from IPython.display import Markdown

if not all((p / "rows.csv").is_file() for p in ISOLATED_OUTPUTS.values()):
    print("Complete both isolated datasets before calculating aggregate results.")
else:
    # Reuse the existing strict provenance, completeness, and pairing checks.
    # This reads saved results only and never launches benchmark inference.
    _, aggregate_coverage = MistakeBenchmarker.isolated_significance(
        ISOLATED_OUTPUTS, tolerance=0.1, n_resamples=9999, seed=0, alpha=0.05,
    )
    aggregate_results = MistakeBenchmarker.isolated_average_table(
        ISOLATED_OUTPUTS, tolerance=0.1,
    )
    display(Markdown("### Combined symbolic results — E1 + E2\n"
                     "Equal-weight mean of each dataset's pooled precision, recall, "
                     "and F1 (%); clean controls excluded."))
    display(aggregate_results.round(3))
    display(aggregate_coverage)

    aggregate_frames = []
    for dataset, output in sorted(ISOLATED_OUTPUTS.items()):
        frame = pd.read_csv(output / "rows.csv", dtype={"group": str})
        frame = frame.loc[
            frame.metric.eq("audio_pitch") & np.isclose(frame.tolerance, 0.1)
            & frame.rate.eq(0.25 if dataset == "E2" else -1.0)
        ]
        aggregate_frames.append(frame)
    aggregate_rows = pd.concat(aggregate_frames, ignore_index=True)
    aggregate_groups = aggregate_rows.groupby(["dataset", "group"]).size().index

    def aggregate_f1(counts):
        numerator = 2 * counts[..., 0]
        denominator = numerator + counts[..., 1] + counts[..., 2]
        return np.divide(numerator, denominator, out=np.zeros_like(numerator),
                         where=denominator > 0).mean(axis=-1)

    aggregate_records = []
    baseline = "Attune (no refinement)"
    for method in aggregate_results.index:
        if method == baseline:
            continue
        # Shape: source clusters x datasets x (TP, FP, FN). A cluster only
        # contributes to its own dataset; F1 is recomputed before averaging.
        paired_counts = []
        for selected_method in (baseline, method):
            grouped = (aggregate_rows.loc[aggregate_rows.method.eq(selected_method)]
                       .groupby(["dataset", "group"])[["tp", "fp", "fn"]]
                       .sum().reindex(aggregate_groups).to_numpy(float))
            by_dataset = np.zeros((len(aggregate_groups), 2, 3))
            for index, dataset in enumerate(sorted(ISOLATED_OUTPUTS)):
                selected_groups = aggregate_groups.get_level_values("dataset") == dataset
                by_dataset[selected_groups, index] = grouped[selected_groups]
            paired_counts.append(by_dataset)
        a, b = paired_counts
        total_a, total_b = a.sum(axis=0), b.sum(axis=0)
        observed = float(aggregate_f1(total_a) - aggregate_f1(total_b))
        exact = 2 ** len(aggregate_groups) <= 9999
        draws = 2 ** len(aggregate_groups) if exact else 9999
        swaps = itertools.product((0, 1), repeat=len(aggregate_groups)) if exact else None
        rng, extreme = np.random.default_rng(0), 0
        for start in range(0, draws, 256):
            size = min(256, draws - start)
            mask = (np.array(list(itertools.islice(swaps, size))) if exact else
                    rng.integers(0, 2, size=(size, len(aggregate_groups))))
            change = (mask @ (b - a).reshape(len(aggregate_groups), -1)).reshape(size, 2, 3)
            differences = aggregate_f1(total_a + change) - aggregate_f1(total_b - change)
            extreme += int(np.count_nonzero(np.abs(differences) >= abs(observed) - 1e-12))
        aggregate_records.append(dict(
            method=method, attune_f1=100 * float(aggregate_f1(total_a)),
            competitor_f1=100 * float(aggregate_f1(total_b)),
            difference_pp=100 * observed,
            p_value=extreme / draws if exact else (extreme + 1) / (draws + 1),
            source_groups=len(aggregate_groups), permutations=draws, exact=exact, seed=0,
        ))
    aggregate_significance = pd.DataFrame(aggregate_records)
    ordered = aggregate_significance.p_value.sort_values()
    adjusted = np.minimum(1, np.maximum.accumulate(
        ordered.to_numpy() * np.arange(len(ordered), 0, -1)))
    aggregate_significance["p_holm"] = pd.Series(adjusted, index=ordered.index)
    aggregate_significance["significant"] = aggregate_significance.p_holm.le(0.05)
    aggregate_significance["alpha"] = 0.05
    aggregate_significance["holm_family_size"] = len(aggregate_significance)
    display(Markdown("**Paired two-sided permutation tests of the combined F1:** "
                     "9,999 swaps (exact when feasible), clustered by dataset/source piece. "
                     "Holm correction across the four combined comparisons, α = 0.05. "
                     "Positive `difference_pp` favors Attune."))
    display(aggregate_significance)


### Combined symbolic results — E1 + E2
Equal-weight mean of each dataset's pooled precision, recall, and F1 (%); clean controls excluded.

,precision,recall,f1
method,,,
Attune (no refinement),98.193,98.476,98.334
Parangonar DualDTW,92.759,91.130,91.936
Nakamura,86.485,97.278,91.317
Parangonar Automatic,84.893,86.226,85.554
Parangonar TheGlueNote,64.120,88.510,74.362


,dataset,tracks,cases,source_groups,methods,excluded_cases
0,E,260,260,98,5,0
1,E2,260,260,256,5,0


**Paired two-sided permutation tests of the combined F1:** 9,999 swaps (exact when feasible), clustered by dataset/source piece. Holm correction across the four combined comparisons, α = 0.05. Positive `difference_pp` favors Attune.

,method,attune_f1,competitor_f1,difference_pp,p_value,source_groups,permutations,exact,seed,p_holm,significant,alpha,holm_family_size
0,Parangonar DualDTW,98.334255,91.935967,6.398288,0.0001,354,9999,False,0,0.0004,True,0.05,4
1,Nakamura,98.334255,91.317370,7.016885,0.0001,354,9999,False,0,0.0004,True,0.05,4
2,Parangonar Automatic,98.334255,85.553625,12.780631,0.0001,354,9999,False,0,0.0004,True,0.05,4
3,Parangonar TheGlueNote,98.334255,74.362143,23.972113,0.0001,354,9999,False,0,0.0004,True,0.05,4


## CocoChorales-E2 — our injection pipeline

The cell below selects all 260 tracks from the frozen E2 cohort, including multiple stems from a source piece. Significance tests keep those stems clustered by source piece. One injection seed (0), one clean control and one 25%-injected performance per track. A seed chooses the random error pattern; using one avoids making a second altered version but leaves the injection rate unchanged. Insertions/deletions shift later notes; the final performance stays monophonic. Truth is recounted after all edits. No symbolic-input baselines or refinement ablations run here.

The run cell appends new tracks and reuses compatible assets/checkpoints. Keep the output folder fixed when increasing `n`; use a new folder if changing evaluation settings. Model setup/downloads are explicit prerequisites, not setup-cell side effects.

In [8]:
E2_OUTPUT = refresh(RESULTS / "coco_e2_final09875_distinct_pieces_seed0_2026-10-08",
                    mode="E2", workers=WORKERS)


E2: refreshing/resuming 520 Attune cases; all competitors cached.
E2: 10/520 Attune cases ready
E2: 20/520 Attune cases ready
E2: 30/520 Attune cases ready
E2: 40/520 Attune cases ready
E2: 50/520 Attune cases ready
E2: 60/520 Attune cases ready
E2: 70/520 Attune cases ready
E2: 80/520 Attune cases ready
E2: 90/520 Attune cases ready
E2: 100/520 Attune cases ready
E2: 110/520 Attune cases ready
E2: 120/520 Attune cases ready
E2: 130/520 Attune cases ready
E2: 140/520 Attune cases ready
E2: 150/520 Attune cases ready
E2: 160/520 Attune cases ready
E2: 170/520 Attune cases ready
E2: 180/520 Attune cases ready
E2: 190/520 Attune cases ready
E2: 200/520 Attune cases ready
E2: 210/520 Attune cases ready
E2: 220/520 Attune cases ready
E2: 230/520 Attune cases ready
E2: 240/520 Attune cases ready
E2: 250/520 Attune cases ready
E2: 260/520 Attune cases ready
E2: 270/520 Attune cases ready
E2: 280/520 Attune cases ready
E2: 290/520 Attune cases ready
E2: 300/520 Attune cases ready
E2: 310/520 A

In [9]:
if E2_OUTPUT is not None and (E2_OUTPUT / "rows.csv").exists():
    metadata = json.loads((E2_OUTPUT / "run.json").read_text())
    assert (
        metadata["status"] == "complete"
    ), "Run incomplete: do not interpret partial results"
    rows = pd.read_csv(E2_OUTPUT / "rows.csv")
    primary = rows[
        rows.method.isin(METHODS)
        & rows.metric.eq("audio_pitch")
        & rows.tolerance.eq(0.1)
        & rows.input.isin(["detected", "audio"])
    ]
    expected_tracks = {c["track_id"] for c in json.loads(COHORT_MANIFESTS["E2"].read_text())["cases"]}
    assert len(expected_tracks) == 260, "Expected the full frozen E2 cohort"
    for method in METHODS:
        for rate in (0.0, 0.25):
            evaluated = primary[primary.method.eq(method) & primary.rate.eq(rate)]
            assert evaluated.case_id.nunique() == 260, (method, rate)
    assert {m["track_id"] for m in metadata["source_metadata"].values()} == expected_tracks
    display(pooled_table(primary[primary.rate.eq(0.25)]))
    display(
        primary[primary.rate.eq(0)]
        .groupby("method")
        .agg(clean_evaluations=("case_id", "nunique"), false_alarms=("fp", "sum"))
    )
    print("Recorded provenance:", E2_OUTPUT / "run.json")
else:
    print("No results yet. Run the preceding benchmark cell.")


/var/folders/_1/wpkj3xlj2sxd981vsh_0fhd80000gn/T/ipykernel_85778/3477852409.py:6: DtypeWarning: Columns (41,42,43) have mixed types. Specify dtype option on import or set low_memory=False.
  rows = pd.read_csv(E2_OUTPUT / "rows.csv")


,Cases,TP,FP,FN,F1 %
method,,,,,
Attune (repeat only),260,629,175,26,86.22
LadderSym,260,123,506,532,19.16
PolyTune,260,88,492,567,14.25


,clean_evaluations,false_alarms
method,,
Attune (repeat only),260,93
LadderSym,260,111
PolyTune,260,154


Recorded provenance: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/evaluation_defaults_c83605debdbad239/run.json


## CocoChorales-E — FluidSynth + MuseScore Synthesized

The cell below selects all 260 tracks from the frozen E (E1) cohort, including multiple stems from a source piece. Significance tests keep those stems clustered by source piece. No new errors are injected into E. `E_BASE/run.json` supplies the historical inference settings only; the saved isolated manifest supplies the track list. Preparation audits the original performed MIDI against `correct + extra` labels, including pitch, timing, velocity and program, then re-renders both audio inputs in parallel. Any discrepancy is recorded; the original performed MIDI is retained. Labels remain byte-identical. Output is 16 kHz mono PCM24; no truth-based boundary corrections.

This preserves author polyphony, which remains challenging for Attune’s monophonic frontend. The inherited `performed_notes` pitch-range policy uses label-derived extrema (±4 semitones) and is oracle-assisted. Pretrained competitor results measure **cross-renderer generalization**. Neither a rendering repair nor a model score proves broad superiority.

The cells below do not benchmark the supplied sequentially rendered audio; that audio is retained for the dedicated audit. CPU and model pools use the worker settings above.

In [10]:
E_OUTPUT = refresh(RESULTS / "coco_e_fluidsynth_final09875_distinct_pieces_seed0_2026-10-08",
                   mode="E", workers=WORKERS)


E: refreshing/resuming 260 Attune cases; all competitors cached.
E: 10/260 Attune cases ready
E: 20/260 Attune cases ready
E: 30/260 Attune cases ready
E: 40/260 Attune cases ready
E: 50/260 Attune cases ready
E: 60/260 Attune cases ready
E: 70/260 Attune cases ready
E: 80/260 Attune cases ready
E: 90/260 Attune cases ready
E: 100/260 Attune cases ready
E: 110/260 Attune cases ready
E: 120/260 Attune cases ready
E: 130/260 Attune cases ready
E: 140/260 Attune cases ready
E: 150/260 Attune cases ready
E: 160/260 Attune cases ready
E: 170/260 Attune cases ready
E: 180/260 Attune cases ready
E: 190/260 Attune cases ready
E: 200/260 Attune cases ready
E: 210/260 Attune cases ready
E: 220/260 Attune cases ready
E: 230/260 Attune cases ready
E: 240/260 Attune cases ready
E: 250/260 Attune cases ready
E: 260/260 Attune cases ready


In [11]:
if E_OUTPUT is not None and (E_OUTPUT / "rows.csv").exists():
    metadata = json.loads((E_OUTPUT / "run.json").read_text())
    assert (
        metadata["status"] == "complete"
    ), "Run incomplete: do not interpret partial results"
    manifest = json.loads((E_OUTPUT / "manifest.json").read_text())
    rows = pd.read_csv(E_OUTPUT / "rows.csv")
    primary = rows[
        rows.protocol.eq("common_pooled_events")
        & rows.metric.eq("audio_pitch")
        & rows.tolerance.eq(0.1)
    ]
    expected = {case["case_id"] for case in manifest["cases"]}
    frozen_tracks = {c["track_id"] for c in json.loads(COHORT_MANIFESTS["E"].read_text())["cases"]}
    assert len(expected) == 260 and expected == frozen_tracks, "Expected the full frozen E cohort"
    for method in ("Attune", "PolyTune", "LadderSym"):
        assert (
            set(primary.loc[primary.method.eq(method), "case_id"]) == expected
        ), method
    display(pooled_table(primary))
    print("Rendering contract:", manifest["intervention"])
    print("Recorded inference provenance:", E_OUTPUT / "run.json")
else:
    print("No results yet. Run the preceding benchmark cell.")


,Cases,TP,FP,FN,F1 %
method,,,,,
Attune,260,361.0,372.0,155.0,57.81
LadderSym,260,252.0,509.0,264.0,39.47
PolyTune,260,172.0,594.0,344.0,26.83


Rendering contract: {'boundary_assistance': False, 'conversion': 'Stereo mean; scipy resample_poly 160/441; mono PCM24 16000 Hz', 'labels': 'Byte-identical native labels; original score and performed MIDI notes unchanged', 'original_run': '/Users/sarah/Desktop/shasha/Attune/benchmarks/results/native_v5_min100_seed0_pieces3/coco', 'renderer': 'Existing NoteBenchmarker.synth_midi: FluidSynth gain 1, reverb/chorus off, 44100 Hz', 'soundfont': '/Users/sarah/Desktop/shasha/Attune/resources/MuseScore_General.sf3', 'soundfont_sha256': '5b85b6c2c61d10b2b91cddd41efcce7b25cd31c8271d511c73afafbef20b6fa3'}
Recorded inference provenance: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/evaluation_defaults_13045cf3f731e4c0/run.json


### End-to-end statistical significance

Paired two-sided permutations of pooled mistake-event F1 at 100 ms / 50 cents, with all stems and seeds from a source piece swapped together. Enumerate all swaps when feasible; otherwise use 9,999 random swaps. One Holm family covers Attune versus PolyTune/LadderSym on re-rendered E and E2 (four comparisons). Clean controls and supplied E audio are excluded. Missing paired cases stop the test. This cell reads saved audio results only; it never launches inference.


In [12]:
audio_outputs = {"E": E_OUTPUT, "E2": E2_OUTPUT}
missing_results = []
for dataset, output in audio_outputs.items():
    if output is None:
        missing_results.append(f"{dataset}: run its audio benchmark cell to select n and load results")
        continue
    required = ("run.json", "rows.csv", "manifest.json") if dataset == "E" else ("run.json", "rows.csv")
    missing = [name for name in required if not (output / name).is_file()]
    if missing:
        missing_results.append(f"{dataset}: missing {', '.join(missing)} in {output}")
    elif json.loads((output / "run.json").read_text()).get("status") != "complete":
        missing_results.append(f"{dataset}: benchmark is not complete in {output}")

if missing_results:
    audio_significance = audio_coverage = None
    print("Significance needs completed E and E2 results for the selected run:")
    print("\n".join(missing_results))
    print("Run the corresponding audio benchmark cell above, then rerun this cell.")
else:
    audio_significance, audio_coverage = MistakeBenchmarker.end_to_end_significance(
        audio_outputs,
        tolerance=0.1, n_resamples=9999, seed=0, alpha=0.05,
    )
    display(audio_coverage)
    display(audio_significance)


/Users/sarah/Desktop/shasha/Attune/benchmarks/modules/mistake/MistakeBenchmarker.py:1267: DtypeWarning: Columns (41,42,43) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(directory / 'rows.csv', dtype={'case_id': str, 'group': str, 'piece': str})


,dataset,tracks,cases,source_groups,methods,excluded_cases
0,E,260,260,98,3,0
1,E2,260,260,256,3,0


,dataset,method,tolerance,tracks,cases,source_groups,attune_f1,competitor_f1,difference_pp,p_value,permutations,exact,seed,p_holm,significant,alpha,holm_family_size
0,E,PolyTune,0.1,260,260,98,57.806245,26.833073,30.973172,0.0001,9999,False,0,0.0004,True,0.05,4
1,E,LadderSym,0.1,260,260,98,57.806245,39.467502,18.338743,0.0001,9999,False,0,0.0004,True,0.05,4
2,E2,PolyTune,0.1,260,260,256,86.223441,14.251012,71.972429,0.0001,9999,False,0,0.0004,True,0.05,4
3,E2,LadderSym,0.1,260,260,256,86.223441,19.158879,67.064562,0.0001,9999,False,0,0.0004,True,0.05,4
